In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt; width:90%;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# 한글 설정
plt.rc('font', family='Malgun Gothic') # 윈도우즈
plt.rc('axes', unicode_minus=False) # 축의 - 깨짐 방지

In [ ]:
df = pd.read_csv('data/일별평균대기오염도_2022(에어코리아).csv', encoding='cp949')
df.head()

In [ ]:
df.info()

In [ ]:
# df['측정일시'] : 20220101 => "2022-01-01" => datetime형으로 형변환(df['측정일'])
df['측정일'] = df['측정일시'].astype(str)
df.head()

In [ ]:
df.info()

In [ ]:
df['측정일'] = df['측정일'].str[:4] + "-" + df['측정일'].str[4:6] + "-" + df['측정일'].str[6:]

In [ ]:
df[['측정일시','측정일']]

In [ ]:
df['측정일'] = pd.to_datetime(df['측정일'])
df.info()

In [ ]:
df.head(1)

In [ ]:
# 현재 컬럼 목록
cols = df.columns.tolist()
cols.remove('측정일')
cols

In [ ]:
new_cols = [cols[0], '측정일'] + cols[1:]
df = df[new_cols]
df.head(1)

In [ ]:
# 서울시 측정소명들
print('측정소명 갯수 : ', df['측정소명'].unique().shape[0])
print(df['측정소명'].unique())

In [ ]:
loc_name = "공항대로"
df_flt = df[df['측정소명']==loc_name]
df_flt.head(2)

In [ ]:
print(f'{loc_name} 데이터(df_flt) 갯수 :', len(df_flt))
print(df_flt.info())

In [ ]:
plt.figure(figsize=(22,6))
plt.plot(df_flt['측정일'], df_flt['미세먼지농도(㎍/㎥)'], label='미세먼지')
plt.plot(df_flt['측정일'], df_flt['초미세먼지농도(㎍/㎥)'], label='초미세먼지')
plt.xlabel('측정일시')
plt.ylabel('먼지농도(㎍/㎥)')
plt.legend(loc="upper left")
plt.show()

In [ ]:
df_flt2 = df_flt[['측정일','미세먼지농도(㎍/㎥)']]
ts = df_flt2.set_index('측정일')
ts.head(10)

In [ ]:
# 시계열 데이터의 구조를 분해해 주는 도구
from statsmodels.tsa.seasonal import seasonal_decompose
result = seasonal_decompose(ts['미세먼지농도(㎍/㎥)'],
                model='additive', # 실제값 추세, 계절성, 잔차를 덧셈으로 분석
                period=30) # 한달 단위로 계정성을 분석해봐
# result : 분석결과
# result.observed : 실제 데이터
# result.trend    : 데이터의 장기적인 변화. 전반적으로 감소, 증가
# result.seasonal : 주기적인 계절성 패턴
# result.resid    : 잔차(실제값에서 추세와 계절성을 뺀 노이즈)
fig, axes = plt.subplots(4,1, figsize=(12,6))
#axes[0].plot(result.observed)
result.observed.plot(ax=axes[0])
axes[1].plot(result.trend)
axes[2].plot(result.seasonal)
axes[3].plot(result.resid)
plt.show()

In [ ]:
def plot_seasonal_decompose(result):
    fig, axes = plt.subplots(4,1, figsize=(12,6))
    result.observed.plot(ax=axes[0])
    axes[0].set_ylabel('관측값')
    axes[1].plot(result.trend)
    axes[1].set_ylabel('트렌드')
    axes[2].plot(result.seasonal)
    axes[2].set_ylabel('계절성')
    axes[3].plot(result.resid)
    axes[3].set_ylabel('잔차')
    plt.xlabel('날짜')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_seasonal_decompose(result)

## seasonal_decompose & RNN/LSTM/GRU & Prophet
      통계적 분석                         예측                     예측
## Prophet
- 계절성, 추세, 휴일효과 등을 자동으로 모델링
- pip install prophet

In [ ]:
df_flt2.head()

In [ ]:
df_flt2.columns = ['ds','y'] #  prophet의 fit시 컬럼명을 ds, y
df_flt2

In [ ]:
from prophet  import Prophet
p_model = Prophet()
p_model.fit(df_flt2)

In [ ]:
# p_model을 이용하여 30일 이후의 데이터를 예측
future = p_model.make_future_dataframe(periods=30)
forecast = p_model.predict(future)
f = p_model.plot(forecast)

In [ ]:
forecast[['ds','yhat', 'yhat_lower', 'yhat_upper']].tail(30)

In [ ]:
f2 = p_model.plot_components(forecast) # 트랜드와 휴일효과 그래프

In [ ]:
forecast.loc[forecast['ds']=='2023-01-16', ['yhat', 'yhat_lower', 'yhat_upper']]